In [5]:
import cv2
from ultralytics import YOLO

def run_detection(
    source="VID00051.AVI", 
    model_path="yolov8n-visdrone.pt", 
    conf_threshold=0.25, 
    target_classes=[0, 1, 3], # 0: pedestrian, 1: people, 3: car
    output_path=None,
    show_live=True
):
    """
    Runs object detection on a video file, IP camera stream, or webcam.
    
    Args:
        source: File path, IP camera stream URL (rtsp://...), or webcam index (0).
        model_path: Name or path to VisDrone model weights.
        conf_threshold: Minimum confidence score for detections.
        target_classes: List of class IDs to filter (e.g., [0, 1, 3] for people & cars).
        output_path: Path to save the output video (optional).
        show_live: Whether to display the video feed in a window.
    """
    # 1. Load the pre-trained VisDrone model
    print(f"Loading model: {model_path}...")
    model = YOLO(model_path)

    # 2. Open video source
    print(f"Opening source: {source}")
    cap = cv2.VideoCapture(source)

    if not cap.isOpened():
        print(f"Error: Could not open video source {source}")
        return

    # Get source properties for output writer if needed
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or fps != fps: # Handle invalid FPS from IP streams
        fps = 25.0 

    out = None
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print("Starting detection. Press 'q' in the video window to quit.")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("End of stream or failed to fetch frame.")
            break

        # 3. Perform object detection with class filtering
        results = model.predict(
            source=frame,
            conf=conf_threshold,
            classes=target_classes,
            verbose=False
        )

        # 4. Annotate frame
        annotated_frame = results[0].plot()

        # 5. Write to output file if specified
        if out is not None:
            out.write(annotated_frame)

        # 6. Display frame (for local GUI environments)
        if show_live:
            cv2.imshow("VisDrone Detection (Cars & People)", annotated_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    # Clean up
    cap.release()
    if out is not None:
        out.release()
    cv2.destroyAllWindows()
    print("Processing complete.")


# ==========================================
# Example Usages:
# ==========================================

if __name__ == "__main__":
    # --- Option A: Process a local video file ---
    # run_detection(
    #     source=r"C:\Users\PC-LAB1\img_processing_lab\videos\day8\VID00051.AVI",
    #     model_path="visdrone.pt",
    #     conf_threshold=0.25,
    #     target_classes=[0, 1, 3], # Pedestrians, People, Cars
    #     output_path="output_detected.mp4",
    #     show_live=True
    # )

    # --- Option B: Live IP Camera Stream (RTSP / HTTP) ---
    ip_stream_url = "rtsp://192.168.1.100:8554/mystream"
    run_detection(
        source=ip_stream_url,
        model_path="visdrone.pt",
        conf_threshold=0.3,
        target_classes=[0, 1, 3],
        show_live=True
    )

Loading model: visdrone.pt...
Opening source: rtsp://192.168.1.100:8554/mystream


KeyboardInterrupt: 

In [6]:
import cv2

rtsp_url = "rtsp://192.168.1.100:8554/mystream"

cap = cv2.VideoCapture(rtsp_url)

if not cap.isOpened():
    print("Failed to open RTSP stream")
    exit()

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to read frame")
        break

    cv2.imshow("RTSP Stream", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

: 